## Import Libraries

In [40]:
import csv
import multiprocessing
import time
from concurrent.futures import ThreadPoolExecutor
from IPython.display import Markdown, display

## Read our CSV Files

In [41]:
with open('./assets/synthetic_sales_data_400k.csv', newline='') as f:
    reader = csv.DictReader(f)
    rows = list(reader)

## Show the first five rows of our files

In [42]:
for row in rows[:5]:
    print(row)

{'Product_ID': '1052', 'Sale_Date': '2023-11-05', 'Sales_Rep': 'David', 'Region': 'East', 'Sales_Amount': '71951.51', 'Quantity_Sold': '48', 'Product_Category': 'Food', 'Unit_Cost': '1147.38', 'Unit_Price': '1594.67', 'Customer_Type': 'Returning', 'Discount': '0.06', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Retail', 'Region_and_Sales_Rep': 'East-David'}
{'Product_ID': '1301', 'Sale_Date': '2024-02-22', 'Sales_Rep': 'Eve', 'Region': 'North', 'Sales_Amount': '8444.02', 'Quantity_Sold': '20', 'Product_Category': 'Electronics', 'Unit_Cost': '451.85', 'Unit_Price': '586.39', 'Customer_Type': 'Returning', 'Discount': '0.28', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Online', 'Region_and_Sales_Rep': 'North-Eve'}
{'Product_ID': '1426', 'Sale_Date': '2023-01-29', 'Sales_Rep': 'Charlie', 'Region': 'East', 'Sales_Amount': '55220.49', 'Quantity_Sold': '36', 'Product_Category': 'Clothing', 'Unit_Cost': '1554.12', 'Unit_Price': '2130.42', 'Customer_Type': 'New', 'Discount': '0.28'

## Number of Data Entries

In [43]:
print(f"No. of Data Rows: {len(rows)}")

No. of Data Rows: 400000


## Sequential / Serial Processes

### Sequential Filtering

In [44]:
start = time.time()
filtered = [row for row in rows if float(row['Sales_Amount']) > 1000]
for row in filtered[:5]:
    print(row)
print(f'Total filtered rows: {len(filtered)}')
filter_time = time.time() - start
print(f"Filtering took {filter_time} seconds")

{'Product_ID': '1052', 'Sale_Date': '2023-11-05', 'Sales_Rep': 'David', 'Region': 'East', 'Sales_Amount': '71951.51', 'Quantity_Sold': '48', 'Product_Category': 'Food', 'Unit_Cost': '1147.38', 'Unit_Price': '1594.67', 'Customer_Type': 'Returning', 'Discount': '0.06', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Retail', 'Region_and_Sales_Rep': 'East-David'}
{'Product_ID': '1301', 'Sale_Date': '2024-02-22', 'Sales_Rep': 'Eve', 'Region': 'North', 'Sales_Amount': '8444.02', 'Quantity_Sold': '20', 'Product_Category': 'Electronics', 'Unit_Cost': '451.85', 'Unit_Price': '586.39', 'Customer_Type': 'Returning', 'Discount': '0.28', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Online', 'Region_and_Sales_Rep': 'North-Eve'}
{'Product_ID': '1426', 'Sale_Date': '2023-01-29', 'Sales_Rep': 'Charlie', 'Region': 'East', 'Sales_Amount': '55220.49', 'Quantity_Sold': '36', 'Product_Category': 'Clothing', 'Unit_Cost': '1554.12', 'Unit_Price': '2130.42', 'Customer_Type': 'New', 'Discount': '0.28'

### Sequential Aggregation (Total Sales Amount per Product_Category)

In [45]:
start = time.time()
aggregated = {}
for row in rows:
    cat = row['Product_Category']
    aggregated[cat] = aggregated.get(cat, 0) + float(row['Sales_Amount'])
for cat, total in aggregated.items():
    print(f'{cat}: {total:.2f}')
agg_time = time.time() - start
print(f"Grouping took {agg_time} seconds")

Food: 7131972595.98
Electronics: 7109945168.08
Clothing: 7138801521.35
Furniture: 7133332433.30
Grouping took 0.28003668785095215 seconds


### Sort by Date (Ascending)

In [46]:
start = time.time()
sorted_rows = sorted(rows, key=lambda r: r['Sale_Date'])
for row in sorted_rows[:5]:
    print(row)
sort_time = time.time() - start
print(f"Sorting took {sort_time} seconds")

{'Product_ID': '1419', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Charlie', 'Region': 'West', 'Sales_Amount': '18217.64', 'Quantity_Sold': '6', 'Product_Category': 'Clothing', 'Unit_Cost': '2179.2', 'Unit_Price': '3264.81', 'Customer_Type': 'New', 'Discount': '0.07', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Retail', 'Region_and_Sales_Rep': 'West-Charlie'}
{'Product_ID': '1410', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Bob', 'Region': 'North', 'Sales_Amount': '66543.25', 'Quantity_Sold': '16', 'Product_Category': 'Food', 'Unit_Cost': '4528.61', 'Unit_Price': '5134.51', 'Customer_Type': 'Returning', 'Discount': '0.19', 'Payment_Method': 'Cash', 'Sales_Channel': 'Online', 'Region_and_Sales_Rep': 'North-Bob'}
{'Product_ID': '1456', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Charlie', 'Region': 'West', 'Sales_Amount': '25519.87', 'Quantity_Sold': '34', 'Product_Category': 'Clothing', 'Unit_Cost': '626.63', 'Unit_Price': '807.08', 'Customer_Type': 'Returning', 'Discount': '0.07', 'Paym

## Parallel Processing

### Parallel Functions

In [47]:
# Filter Function
def filter_chunk(chunk):
    return [row for row in chunk if float(row['Sales_Amount']) > 1000]
# Aggregation Function
def agg_chunk(chunk):
    result = {}
    for row in chunk:
        cat = row['Product_Category']
        result[cat] = result.get(cat, 0) + float(row['Sales_Amount'])
    return result
# Sort Function
def sort_chunk(chunk):
    return sorted(chunk, key=lambda r: r['Sale_Date'])
# Parallel Processing Function
def parallel_operation(data, func):
    num_workers = multiprocessing.cpu_count()
    size = len(data) // num_workers
    chunks = [data[i*size:(i+1)*size] for i in range(num_workers)]
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        results = list(executor.map(func, chunks))
    return results

### Parallel Filtering

In [48]:
start = time.time()
filtered_chunks = parallel_operation(rows, filter_chunk)
filtered_parallel = [row for chunk in filtered_chunks for row in chunk]
filter_time_p = time.time() - start
for row in filtered_parallel[:5]:
    print(row)
print(f'Total filtered rows: {len(filtered_parallel)}')
print(f"Parallel filtering took {filter_time_p} seconds")

{'Product_ID': '1052', 'Sale_Date': '2023-11-05', 'Sales_Rep': 'David', 'Region': 'East', 'Sales_Amount': '71951.51', 'Quantity_Sold': '48', 'Product_Category': 'Food', 'Unit_Cost': '1147.38', 'Unit_Price': '1594.67', 'Customer_Type': 'Returning', 'Discount': '0.06', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Retail', 'Region_and_Sales_Rep': 'East-David'}
{'Product_ID': '1301', 'Sale_Date': '2024-02-22', 'Sales_Rep': 'Eve', 'Region': 'North', 'Sales_Amount': '8444.02', 'Quantity_Sold': '20', 'Product_Category': 'Electronics', 'Unit_Cost': '451.85', 'Unit_Price': '586.39', 'Customer_Type': 'Returning', 'Discount': '0.28', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Online', 'Region_and_Sales_Rep': 'North-Eve'}
{'Product_ID': '1426', 'Sale_Date': '2023-01-29', 'Sales_Rep': 'Charlie', 'Region': 'East', 'Sales_Amount': '55220.49', 'Quantity_Sold': '36', 'Product_Category': 'Clothing', 'Unit_Cost': '1554.12', 'Unit_Price': '2130.42', 'Customer_Type': 'New', 'Discount': '0.28'

### Parallel Aggregation

In [49]:
start = time.time()
agg_chunks = parallel_operation(rows, agg_chunk)
agg_parallel = {}
for chunk_result in agg_chunks:
    for cat, total in chunk_result.items():
        agg_parallel[cat] = agg_parallel.get(cat, 0) + total
agg_time_p = time.time() - start
for cat, total in agg_parallel.items():
    print(f'{cat}: {total:.2f}')
print(f"Parallel grouping took {agg_time_p} seconds")

Food: 7131972595.98
Electronics: 7109945168.08
Clothing: 7138801521.35
Furniture: 7133332433.30
Parallel grouping took 0.23199701309204102 seconds


### Parallel Sorting

In [50]:
start = time.time()
sorted_chunks = parallel_operation(rows, sort_chunk)
sorted_parallel = [row for chunk in sorted_chunks for row in chunk]
sort_time_p = time.time() - start
for row in sorted_parallel[:5]:
    print(row)
print(f'Total sorted rows: {len(sorted_parallel)}')
print(f"Parallel sorting took {sort_time_p} seconds")

{'Product_ID': '1419', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Charlie', 'Region': 'West', 'Sales_Amount': '18217.64', 'Quantity_Sold': '6', 'Product_Category': 'Clothing', 'Unit_Cost': '2179.2', 'Unit_Price': '3264.81', 'Customer_Type': 'New', 'Discount': '0.07', 'Payment_Method': 'Credit Card', 'Sales_Channel': 'Retail', 'Region_and_Sales_Rep': 'West-Charlie'}
{'Product_ID': '1410', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Bob', 'Region': 'North', 'Sales_Amount': '66543.25', 'Quantity_Sold': '16', 'Product_Category': 'Food', 'Unit_Cost': '4528.61', 'Unit_Price': '5134.51', 'Customer_Type': 'Returning', 'Discount': '0.19', 'Payment_Method': 'Cash', 'Sales_Channel': 'Online', 'Region_and_Sales_Rep': 'North-Bob'}
{'Product_ID': '1456', 'Sale_Date': '2023-01-01', 'Sales_Rep': 'Charlie', 'Region': 'West', 'Sales_Amount': '25519.87', 'Quantity_Sold': '34', 'Product_Category': 'Clothing', 'Unit_Cost': '626.63', 'Unit_Price': '807.08', 'Customer_Type': 'Returning', 'Discount': '0.07', 'Paym

## Execution Time Summary

In [54]:
num_workers = multiprocessing.cpu_count()
md = f"""
| Operation | Sequential (s) | Parallel (s) | Speedup | Efficiency
|---|---|---|---|---|
| Filtering | {filter_time} | {filter_time_p} | {filter_time / filter_time_p} | {(filter_time / filter_time_p)/num_workers}|
| Aggregation | {agg_time} | {agg_time_p} | {agg_time / agg_time_p} |{(agg_time / agg_time_p)/num_workers}|
| Sorting | {sort_time} | {sort_time_p} | {sort_time / sort_time_p} |{(sort_time / sort_time_p)/num_workers}|
"""
display(Markdown(md))


| Operation | Sequential (s) | Parallel (s) | Speedup | Efficiency
|---|---|---|---|---|
| Filtering | 0.520998477935791 | 0.24605655670166016 | 2.1173931917103666 | 0.5293482979275916|
| Aggregation | 0.28003668785095215 | 0.23199701309204102 | 1.2070702295630513 |0.30176755739076283|
| Sorting | 0.5109989643096924 | 0.7100436687469482 | 0.7196725874782877 |0.17991814686957192|
